# Original course-project exploration
Spring 2025 team project. Hongjia Liu was the primary contributor to implementation and experiments.
This notebook preserves historical code with outputs cleared. It is not the revised pipeline.
Known evaluation limitations and required external data/caches are described in ../docs/PROJECT_REPORT.md.
Run revised experiments from src/pipeline.py at the repository root.


In [ ]:
import os, glob, json, time, math, random, pickle
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict, OrderedDict
from tqdm import tqdm
from scipy.interpolate import RegularGridInterpolator
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import matplotlib.pyplot as plt
import seaborn as sns
from model.SWIN_trans import SwinTransformer  

In [ ]:
# ──────────────────────────────────────────────────────────────
# 0. 常量与通道→网格映射  (保持不变，只把空格填 NaN)
# ──────────────────────────────────────────────────────────────
DATA_DIR  = './data'
CACHE_DIR = './cache3_all'
SAVE_ROOT  = './runs_loso'
FRAME     = 4
BANDS     = ['delta','theta','alpha','beta','gamma']
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
Path(SAVE_ROOT).mkdir(exist_ok=True)

os.makedirs(CACHE_DIR, exist_ok=True)
CH_NAMES = ['Fp1','Fpz','Fp2','AF7','AF3','AF4','AF8',
            'F7','F5','F3','F1','Fz','F2','F4','F6','F8',
            'FT7','FC5','FC3','FC1','FCz','FC2','FC4','FC6','FT8',
            'T7','C5','C3','C1','Cz','C2','C4','C6','T8',
            'TP7','CP5','CP3','CP1','CPz','CP2','CP4','CP6','TP8',
            'P7','P5','P3','P1','Pz','P2','P4','P6','P8',
            'PO7','PO3','POz','PO4','PO8','O1','Oz','O2','F9','F10','TP9','TP10']

# 手动把 64 电极放进 9×9，示例只列两行——生产环境请补全
GRID9 = np.full((9, 9), None, dtype=object)      # 先全部置空

# 每行 list = {(row, col): 'ChanName'}
coords = {
    # row 0  ──────────────────────────────────────────────
    'F9':  (0,0),  'Fp1':(0,2),  'Fpz':(0,4),  'Fp2':(0,6),  'F10':(0,8),

    # row 1
    'AF7': (1,1),  'AF3':(1,3),  'AF4':(1,5),  'AF8':(1,7),

    # row 2
    'F7':  (2,0),  'F5': (2,1),  'F3': (2,2),  'F1': (2,3),  'Fz': (2,4),
    'F2':  (2,5),  'F4': (2,6),  'F6': (2,7),  'F8': (2,8),

    # row 3
    'FT7': (3,0),  'FC5':(3,1),  'FC3':(3,2),  'FC1':(3,3),  'FCz':(3,4),
    'FC2': (3,5),  'FC4':(3,6),  'FC6':(3,7),  'FT8':(3,8),

    # row 4
    'T7':  (4,0),  'C5': (4,1),  'C3': (4,2),  'C1': (4,3),  'Cz': (4,4),
    'C2':  (4,5),  'C4': (4,6),  'C6': (4,7),  'T8': (4,8),

    # row 5
    'TP7': (5,0),  'CP5':(5,1),  'CP3':(5,2),  'CP1':(5,3),  'CPz':(5,4),
    'CP2': (5,5),  'CP4':(5,6),  'CP6':(5,7),  'TP8':(5,8),

    # row 6
    'P7':  (6,0),  'P5': (6,1),  'P3': (6,2),  'P1': (6,3),  'Pz': (6,4),
    'P2':  (6,5),  'P4': (6,6),  'P6': (6,7),  'P8': (6,8),

    # row 7
    'PO7': (7,1),  'PO3':(7,3),  'POz':(7,4),  'PO4':(7,5),  'PO8':(7,7),

    # row 8
    'TP9': (8,0),  'O1': (8,2),  'Oz': (8,4),  'O2': (8,6),  'TP10':(8,8),
}
grid9 = np.full((9, 9, FRAME), np.nan, dtype=np.float32)
idx2rc = {CH_NAMES.index(ch): (r, c) for ch, (r, c) in coords.items()}

for ch,(r,c) in coords.items():            # coords 与你之前相同
    idx2rc[CH_NAMES.index(ch)] = (r, c)


In [ ]:

def topo_map(mat64x4):
    """64×4 → (4,32,32) cubic 插值"""
    grid9[:] = np.nan
    for ch,(r,c) in idx2rc.items(): grid9[r,c]=mat64x4[ch]
    filled = np.nan_to_num(grid9, nan=np.nanmean(grid9))
    y,x = np.mgrid[0:8:32j,0:8:32j]
    return torch.from_numpy(
        scipy.interpolate.RegularGridInterpolator((range(9),)*2, filled,
                      method='cubic')(np.stack([y,x],-1))).permute(2,0,1)


In [ ]:
def preprocess_subject(sub:int, ses:str):
    out = Path(CACHE_DIR)/f'sub{sub:02d}_{ses}.npz'
    if out.exists(): return
    src = f'{DATA_DIR}/sub-{sub:02d}/sub-{sub:02d}_ses-{ses}_task-emotion_de.npy'
    de  = np.load(src,mmap_mode='r').astype(np.float32)          # (64,630,5)
    de  = de.reshape(64,21,30,5)[...,:28,:]                      # →(64,21,7,4,5)
    feats,labels=[],[]
    map7to3=np.array([0,0,0,1,2,2,2],np.int64)
    trial_lbl=np.repeat(np.arange(7),3)
    for t,b in itertools.product(range(21),range(7)):
        # (5,4,32,32)
        topo5=torch.stack([topo_map(de[...,band][:,t,b]) for band in range(5)])
        feats.append(topo5.numpy())
        labels.append(map7to3[trial_lbl[t]])
    np.savez_compressed(out,x=np.stack(feats),y=np.array(labels,np.int64))


In [ ]:
# 删除缓存 cache3

# import shutil, pathlib
# shutil.rmtree(pathlib.Path("./cache3"), ignore_errors=True)

# # Windows Powershell
# Remove-Item -Path .\cache3 -Recurse -Force



In [ ]:
TRAIN_STATS = {}
class EmoEEGDataset(torch.utils.data.Dataset):
    def __init__(self,
                 leave_sub,
                 context='ima',
                 split='train',
                 band=None):
        """
        band = 0-4  时：返回对应单频带 (1,4,32,32)
        band = None 时：返回所有 5 带  (5,4,32,32)
        """
        # ---------- 读缓存 ----------
        if context == 'multi':
            fns = glob.glob(f'{CACHE_DIR}/sub*_[iv][ma][ad].npz')
        else:
            fns = glob.glob(f'{CACHE_DIR}/sub*_{context}.npz')

        xs, ys, subs = [], [], []
        for fn in sorted(fns):
            sid = int(fn[-10:-8])
            d   = np.load(fn, mmap_mode='r')
            xs.append(d['x'][:, band] if band is not None else d['x'])
            ys.append(d['y'])
            subs.append(np.full_like(d['y'], sid))

        X, Y, S = map(np.concatenate, (xs, ys, subs))
        msk = S != leave_sub if split == 'train' else S == leave_sub
        self.x = torch.tensor(X[msk]).float()
        self.y = torch.tensor(Y[msk]).long()

        # ---------- z-score ----------
        key = (leave_sub, context)
        if split == 'train':
            mu = self.x.mean((0, *range(2, self.x.ndim)), keepdim=True)
            sd = self.x.std ((0, *range(2, self.x.ndim)), keepdim=True) + 1e-8
            TRAIN_STATS[key] = (mu, sd)
        self.mu, self.sd = TRAIN_STATS[key]

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        x = (self.x[idx] - self.mu) / self.sd
        x = x.squeeze()
        return x, self.y[idx]


In [ ]:
from model.SWIN_trans import SwinTransformer   # 原论文实现

def build_swin(in_chans=4, num_classes=3):
    return SwinTransformer(
            # Model parameter
            img_size=32,
            patch_size=4,
            in_chans=in_chans, num_classes=num_classes,
            embed_dim=96,
            depths=[2, 2, 6, 2],  
            num_heads=[3, 6, 12, 24],
            window_size=2,
            mlp_ratio=4., qkv_bias=True,
            drop_rate=0.107, attn_drop_rate=0.07, drop_path_rate=0.07,
            patch_norm=True,
            use_checkpoint=False
    )


class BandFuseSwin(nn.Module):
    def __init__(self,in_bands=5,fuse_ch=32,n_cls=3):
        super().__init__()
        self.band_mix = nn.Sequential(
            nn.Conv2d(in_bands,fuse_ch,kernel_size=1,bias=False),
            nn.GroupNorm(8,fuse_ch),
            nn.GELU()
        )
        self.backbone = SwinTransformer(
            img_size=32,patch_size=4,in_chans=fuse_ch,num_classes=n_cls,
            embed_dim=96,depths=[2,2,6,2],num_heads=[3,6,12,24],
            window_size=2,mlp_ratio=4.,qkv_bias=True,
            drop_rate=0.07,attn_drop_rate=0.07,drop_path_rate=0.07,
            patch_norm=True)

    def forward(self,x):
        return self.backbone(self.band_mix(x))


In [ ]:
def plot_cm(cm, title, save_path):
    plt.figure(figsize=(3,3))
    sns.heatmap(cm, annot=True, fmt='d',
                cmap='Blues', cbar=False,
                xticklabels=['0','1','2'],
                yticklabels=['0','1','2'])
    plt.title(title); plt.ylabel('True'); plt.xlabel('Pred')
    plt.tight_layout(); plt.savefig(save_path); plt.close()


In [ ]:
cls_w    = torch.tensor([1.,1.6,1.], device=device)

def train_subject(test_sub, ctx, band, epochs=60, batch=256):
    tr_ds = EmoEEGDataset(test_sub, ctx, 'train', band)
    # TRAIN_STATS[(test_sub,ctx)] = tr_ds._stat  # 原有逻辑
    te_ds = EmoEEGDataset(test_sub, ctx, 'test',  band)

    tr_ld = DataLoader(tr_ds, batch, shuffle=True, pin_memory=True)
    te_ld = DataLoader(te_ds, batch, pin_memory=True)
    net = build_swin().to(device)
    # net    = BandFuseSwin().to(device)         # ← 只改这一行即可

    # class_counts = torch.bincount(tr_ds.y, minlength=3)
    # weights = (class_counts.sum() / class_counts);  weights = weights / weights.sum() * len(weights)
    # weights = weights.to(device).float()
    # criterion = torch.nn.CrossEntropyLoss(weight=weights)

    head, back = [], []
    for n,p in net.named_parameters():
        (head if n.startswith('backbone.head') or n.startswith('head')
         else back).append(p)
    opt  = torch.optim.AdamW(
              [{'params': back,'lr':1e-5},
               {'params': head,'lr':5e-5}],
              weight_decay=1e-3)
    scaler  = torch.amp.GradScaler('cuda')
    best_acc, best_cm, best_state = 0, None, None

    for ep in range(1, epochs+1):
        # ─── train ───
        net.train(); loss_sum = 0; n_batch = 0
        for x,y in tr_ld:
            x,y = x.to(device), y.to(device)
            with torch.amp.autocast('cuda'):
                logits = net(x)
                loss   = F.cross_entropy(logits, y, weight=cls_w)
                # loss   = criterion(logits, y)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            loss_sum += loss.item(); n_batch += 1

        # ─── validate ───
        net.eval(); y_true=[]; y_pred=[]
        with torch.no_grad():
            for x,y in te_ld:
                p = net(x.to(device)).argmax(1).cpu()
                y_true.extend(y); y_pred.extend(p)
        y_true = torch.tensor(y_true); y_pred = torch.tensor(y_pred)
        acc   = (y_true==y_pred).float().mean().item()
        if acc > best_acc:
            best_acc, best_state = acc, net.state_dict()
            best_cm = confusion_matrix(y_true, y_pred)

        # ─── 实时打印 ───
        if ep%5==0 or ep==EPOCHS:
            print(f'[{ctx}-{BANDS[band]}] Sub{test_sub:02d} '
                f'Ep{ep:03d}  train-loss={loss_sum/n_batch:.4f} '
                f'acc={acc:.3f}')
              
    # ─── 训练结束：保存最佳权重 + 混淆矩阵图 ───
    sub_dir = Path(SAVE_ROOT, f'{ctx}-{BANDS[band]}', f'sub{test_sub:02d}')
    sub_dir.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, sub_dir/'best.pth')
    plot_cm(best_cm, f'Sub{test_sub:02d}  acc={best_acc:.3f}',
            sub_dir/'confusion.png')

    return best_cm, best_acc 

In [ ]:
# ────────── 基本配置 ──────────
LABELS       = ['neg', 'neu', 'pos']
CTX_OPTIONS  = ['vid', 'ima', 'multi']      # 3 种情境
BAND_OPTIONS = list(range(5))               # 0-δ … 4-γ
BAND_NAMES   = ['delta', 'theta', 'alpha', 'beta', 'gamma']    # 用于横坐标显示

SUBJECTS     = range(15, 16)                # 你当前想跑的受试者
EPOCHS       = 40
BATCH_SIZE   = 64

CTX_COLORS = {                              # 柱子配色
    'vid'  : '#c7e3c1',   # 淡绿
    'ima'  : '#f4b082',   # 淡橙
    'multi': '#d2d9f5'    # 淡蓝
}

RUNS_DIR = Path('runs_loso2')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
# ────────────────────────────

if __name__ == '__main__':

    # 0) 预处理（每个 sub × ses 各保存一次 5-band 缓存）
    for sid in tqdm(SUBJECTS, desc='preprocess'):
        for ses in ('ima', 'vid'):
            preprocess_subject(sid, ses)

    # 1) 训练并收集 (ctx, band) 准确率
    acc_dict   = {ctx: {b: [] for b in BAND_OPTIONS} for ctx in CTX_OPTIONS}
    best_cm_all_subjects = []          # 用于汇总 grand CM

    for sid in SUBJECTS:
        best_acc_per_sub, best_cm_per_sub = 0.0, None

        for ctx in CTX_OPTIONS:
            for band in BAND_OPTIONS:
                cm, acc = train_subject(sid, ctx, band,
                                        epochs=EPOCHS, batch=BATCH_SIZE)
                acc_dict[ctx][band].append(acc)

                # 累加 grand CM
                if cm is not None:
                    best_cm_all_subjects.append(cm)

                # 记录该 subject 的 global best
                if acc > best_acc_per_sub:
                    best_acc_per_sub, best_cm_per_sub = acc, cm

        # 也可以只累加每个 subject 的 global best：
        # best_cm_all_subjects.append(best_cm_per_sub)

    # 2) 画柱状图：五频段 × 三情境
    fig, ax = plt.subplots(figsize=(8, 4))
    width   = 0.22                        # 三组柱子的宽度
    x_pos   = np.arange(len(BAND_OPTIONS))

    for i, ctx in enumerate(CTX_OPTIONS):
        means = [np.mean(acc_dict[ctx][b]) for b in BAND_OPTIONS]
        ax.bar(x_pos + (i-1)*width, means, width,
               label=ctx,
               color=CTX_COLORS[ctx],
               edgecolor='black', linewidth=0.8)
        # 在柱顶标注数值
        for x, m in zip(x_pos + (i-1)*width, means):
            ax.text(x, m + 0.01, f'{m:.2f}',
                    ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(BAND_NAMES, fontsize=12)
    ax.set_ylabel('Accuracy')
    ax.set_xlabel('Frequency Band')
    ax.set_ylim(0, 1.0)
    ax.axhline(1/3, ls='--', c='k', lw=1, label='Chance 33 %')
    ax.legend()
    ax.set_title('Ternary Classification Cross Subject')
    plt.tight_layout()
    plt.savefig(RUNS_DIR / 'bar_acc_ctx_band.png', dpi=120)
    plt.close()

    # 3) 汇总 grand CM → 百分比热图
    grand_cm = sum(best_cm_all_subjects)
    mean_best_acc = np.mean([
        max(acc_dict[ctx][b][i]
            for ctx in CTX_OPTIONS for b in BAND_OPTIONS)
        for i in range(len(SUBJECTS))
    ])

    plot_cm(grand_cm,
            title=(f'Overall CM (mean best acc={mean_best_acc:.3f})'),
            fname=RUNS_DIR / 'cm_overall_best.png',
            as_percent=True)

    # 4) 控制台输出
    print('\n========== FINISHED ==========')
    for ctx in CTX_OPTIONS:
        for b in BAND_OPTIONS:
            print(f'{ctx}-{BAND_NAMES[b]}: '
                  f'{np.mean(acc_dict[ctx][b]):.3f}')
    print(f'总体平均最佳 acc: {mean_best_acc:.3f}')
